# All dMRI and tensor-domain interpolation demos

Consolidated entry point. Raw-signal methods and tensor-domain methods remain explicitly separated.

## 1. Prepare the Colab environment

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

PROJECT_ROOT = Path('/content/dMRI_Interpolation_Repository')
DATA_ROOT = Path('/content/dMRI_Data')
if not PROJECT_ROOT.is_dir():
    from google.colab import files
    print('Upload dMRI_Interpolation_Repository.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Project directory not found: {PROJECT_ROOT}')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    from google.colab import files
    print('Upload dMRI_Data.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No dMRI data ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    raise FileNotFoundError(f'dMRI data folder not found: {DATA_ROOT}')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(PROJECT_ROOT / 'requirements_colab.txt')
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project ready:', PROJECT_ROOT)
print('External dMRI data ready:', DATA_ROOT)

## 2. Validate core numerical functions

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, str(PROJECT_ROOT / 'run_core_tests.py')])

## 3. Raw dMRI: complete-slice nearest neighbor, bilinear, and patch GP

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dmri_io import load_dmri_study
from dmri_interpolation import reconstruct_dmri
from dmri_gp import patch_gp_upsample

study = load_dmri_study(PROJECT_ROOT)
raw = reconstruct_dmri(study.data, factor=2, slice_indices=[19], gradient_indices=[15])
reference = raw['reference'][:, :, 0, 0]
for method in ('nearest', 'bilinear'):
    estimate = raw[method][:, :, 0, 0]
    print(method, 'RMSE =', float(np.sqrt(np.mean((estimate-reference)**2))))
low = raw['low_resolution'][:, :, 0, 0]
gp = patch_gp_upsample(
    low, reference.shape, patch_size=16, factor=2, stride=14,
    blending='hann', preserve_samples=True, optimize_kernel=False,
    show_progress=True, return_details=True,
)
gp_slice = gp['reconstructed']
evaluation_mask = np.ones(reference.shape, dtype=bool)
evaluation_mask[::2, ::2] = False
print('GP joined patches =', gp['patch_count'])
print('GP unsampled RMSE =', float(np.sqrt(np.mean(
    (gp_slice[evaluation_mask]-reference[evaluation_mask])**2
))))

## 4. Raw dMRI: complete-slice CNN reconstruction

In [ ]:
from dmri_cnn import cnn_reconstruct_slice
cnn = cnn_reconstruct_slice(
    reference, patch_size=16, factor=2, stride=14,
    blending='hann', preserve_samples=True,
    epochs_per_patch=5, show_progress=True,
)
cnn_slice = cnn['reconstructed']
print('CNN joined patches =', cnn['patch_count'])
print('CNN unsampled RMSE =', cnn['rmse_unsampled'])

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
panels = (
    (reference, 'Reference', 'gray'),
    (raw['nearest'][:, :, 0, 0], 'Nearest neighbor', 'gray'),
    (raw['bilinear'][:, :, 0, 0], 'Bilinear', 'gray'),
    (gp_slice, 'GP full slice', 'gray'),
    (cnn_slice, 'CNN full slice', 'gray'),
    (np.abs(cnn_slice-reference), 'CNN absolute error', 'magma'),
)
for axis, (image, title, cmap) in zip(axes.ravel(), panels):
    shown = axis.imshow(image.T, cmap=cmap, origin='lower')
    axis.set_title(title)
    axis.axis('off')
    if 'error' in title.lower():
        fig.colorbar(shown, ax=axis, fraction=0.046)
plt.tight_layout()

## 5. Tensor domain: GWP for order 2

In [ ]:
from gwp_tensor import gwp_interpolate
from tensor_utils import frobenius_error, load_tensor_field, unsampled_mask
tensor_root = PROJECT_ROOT / 'data' / 'tensors'
tensor2 = load_tensor_field(tensor_root / 'training_data2.mat')
gwp = gwp_interpolate(tensor2[:, ::2, ::2], tensor2.shape[1:], factor=2)
gwp_error = frobenius_error(
    gwp['coefficients'], tensor2, 2, unsampled_mask(tensor2.shape[1:], 2)
)
print('GWP:', gwp_error['mean'], '+/-', gwp_error['std'])

## 6. Tensor domain: CDP for orders 2, 4, and 6

In [ ]:
from cdp_tensor import FAST_RANKS, cdp_interpolate
for order in (2, 4, 6):
    field = load_tensor_field(tensor_root / f'training_data{order}.mat')
    cdp = cdp_interpolate(
        field[:, ::2, ::2], field.shape[1:], order=order, factor=2,
        rank=FAST_RANKS[order], cp_iterations=80,
    )
    error = frobenius_error(
        cdp['coefficients'], field, order, unsampled_mask(field.shape[1:], 2)
    )
    print(f"CDP order {order}: {error['mean']} +/- {error['std']}")

## 7. DTI and FA from the raw dMRI study

In [ ]:
from dmri_dti import compute_fa, fit_dti
dti = fit_dti(study.data, study.bvals, study.bvecs, slice_indices=[19])
fa = compute_fa(dti['coefficients'])['fa'][:, :, 0]
print('FA range:', float(fa.min()), float(fa.max()))
plt.figure(figsize=(5, 5)); plt.imshow(fa.T, cmap='viridis', origin='lower', vmin=0, vmax=1)
plt.colorbar(label='FA'); plt.title('Reference FA'); plt.axis('off');